# 03 — Version Selection
Picks the best semantic-extraction run per page. Both designs kept below.


In [ ]:
import re
import pandas as pd
import io
import csv
import shutil
import glob
import json
import time
import tempfile
from pathlib import Path
from datetime import datetime
from langchain_openai import ChatOpenAI

#drive
from drive_utils import (
    get_drive_service,
    find_folder,
    get_or_create_folder,
    list_files_in_folder,
    download_text,
    upload_text,
    find_latest_folder_local,
    find_latest_folder_drive,
)

from config import (
    EVAL_RESULTS_CSV,       # fallback if no eval folder found
    EVAL_SUMMARY_CSV,       # fallback if no eval folder found
    EVAL_OUTPUT_DIR,        # parent dir of timestamped eval subfolders
    CLEAN_PAGES_DIR,
    OCR_RESULTS_DIR,
    LOGS_DIR,
    OPENAI_KEY_FILE,
    DRIVE_ROOT_FOLDER,
    DRIVE_OCR_FOLDER,
    DRIVE_CLEAN_PAGES_FOLDER,
    SAVE_MODE,
)


In [ ]:
#general helpers

# Excludes a run for a missing file, zero rows, or 100% missing address.
def is_excluded(eval_row):
    if eval_row is None or len(eval_row) == 0:
        return False, ""
    issues      = str(eval_row.get("Issues") or "")
    rows        = float(eval_row.get("Rows") or 0)
    missing_pct = extract_missing_pct(eval_row.get("Propagation Note"))

    if "EMPTY_FILE" in issues:
        return True, "EMPTY_FILE"
    if rows == 0:
        return True, "zero_rows"
    if missing_pct == 100.0:
        return True, "100pct_missing_address"
    return False, ""

# Pulls the missing-address percentage out of a propagation note.
def extract_missing_pct(note):
    if pd.isna(note) or not note:
        return 0.0
    m = re.search(r'\((\d+)%\)', str(note))
    return float(m.group(1)) if m else 0.0


# Average pairwise value-similarity across a page's runs.
def compute_avg_value_sim(page_name, compare_df):
    page_compares = compare_df[compare_df["Page"] == page_name]
    sims = page_compares["Value Similarity %"].dropna().tolist()
    return sum(sims) / len(sims) if sims else None


# RUN PAIRING
# page_X_semantic_N.csv pairs with page_X_ocr_N.txt by run number N.
# If two files share the same N, keeps the most recent by createdTime.

def pair_runs(semantic_files, ocr_files):
    # pairs semantic CSVs with OCR txts by run number
    def build_run_map(files, pattern):
        run_map = {}
        for f in files:
            m = re.search(pattern, f["name"])
            if not m:
                continue
            n = int(m.group(1))
            if n not in run_map or f["createdTime"] > run_map[n]["createdTime"]:
                run_map[n] = f
        return run_map

    sem_map = build_run_map(semantic_files, r'_semantic_(\d+)\.csv$')
    ocr_map = build_run_map(ocr_files,      r'_ocr_(\d+)\.txt$')

    paired = []
    for n in sorted(sem_map.keys()):
        paired.append({
            "run_num":       n,
            "semantic_file": sem_map[n],
            "ocr_file":      ocr_map.get(n),
        })
    return paired


# SCORING — per-page semantic CSVs

def score_run(eval_row, avg_value_sim, csv_text=None):
    # scores a run 0-100: address coverage, schema, column diversity, name completeness, row count, run agreement, GT metrics
    if eval_row is None or len(eval_row) == 0:
        return 0.0

    rows        = float(eval_row.get("Rows") or 0)
    missing_pct = extract_missing_pct(eval_row.get("Propagation Note"))
    f1          = eval_row.get("semantic_f1")
    char_acc    = eval_row.get("char_accuracy")
    bow         = eval_row.get("bow_similarity")
    issues      = str(eval_row.get("Issues") or "")

    # Address coverage — most important signal for geocoding downstream
    score  = (1 - missing_pct / 100.0) * 30

    # Schema consistency — penalise missing expected columns
    score += 0 if "missing_cols" in issues else 15

    # Row count — normalised against 200 as a generous ceiling
    score += min(rows / 200.0, 1.0) * 8

    # Value similarity — agreement with other runs
    score += ((avg_value_sim / 100.0) * 10) if avg_value_sim is not None else 5

    # Ground truth metrics
    score += ((float(f1)       / 100.0) * 10) if pd.notna(f1)       else 5
    score += ((float(char_acc) / 100.0) *  4) if pd.notna(char_acc) else 0
    score += ((float(bow)      / 100.0) *  3) if pd.notna(bow)      else 0

    # Column diversity and name completeness — require reading CSV text
    # Combined 20 pts: 12 for diversity, 8 for name completeness
    if csv_text:
        try:
            lines = [l for l in csv_text.splitlines() if l.strip()]
            if len(lines) >= 2:
                rows_parsed = list(csv.DictReader(io.StringIO(csv_text)))
                if rows_parsed:
                    header = list(rows_parsed[0].keys())

                    # Column diversity: average fill rate across all columns
                    col_fill_rates = []
                    for col in header:
                        filled = sum(
                            1 for r in rows_parsed
                            if r.get(col, "").strip()
                        )
                        col_fill_rates.append(filled / max(len(rows_parsed), 1))
                    avg_fill = sum(col_fill_rates) / max(len(col_fill_rates), 1)
                    score += avg_fill * 12

                    # Name completeness: names with space (first+last) and length > 5
                    name_col = next(
                        (h for h in header if h.strip().lower() == "name"), None
                    )
                    if name_col:
                        name_vals = [
                            r.get(name_col, "").strip()
                            for r in rows_parsed
                            if r.get(name_col, "").strip()
                            and r.get(name_col, "").strip() != "__SECTION__"
                        ]
                        if name_vals:
                            complete = sum(
                                1 for n in name_vals
                                if " " in n and len(n) > 5
                            )
                            score += (complete / len(name_vals)) * 8
        except Exception:
            pass
    else:
        # No CSV text available — give neutral half-credit for diversity/completeness
        score += 10

    return round(min(score, 100.0), 2)


# Reads a file from local path or Drive depending on save_mode.
# file_meta must have "name" and either "local_path" or "id".
def read_file(service, file_meta, save_mode):
    # reads a file from local or Drive per save_mode, local first for 'both'
    if save_mode in ("local", "both") and file_meta.get("local_path"):
        p = Path(file_meta["local_path"])
        if p.exists():
            return p.read_text(encoding="utf-8")
        if save_mode == "local":
            return None
        # "both" and local not found — fall through to Drive

    if save_mode in ("drive", "both") and file_meta.get("id"):
        return download_text(service, file_meta["id"])

    return None


## Original design — LLM arbitration (superseded)


Tier 1 scoring, Tier 2 LLM arbitration on close calls. Kept for the before/after comparison.

For each page with multiple semantic extraction runs, picks the best one and writes it to clean_pages/page_X/ as page_X_semantic.csv. The matching OCR .txt is also saved.

In [ ]:
#  INPUTS:
#    - evaluation output folder (auto-detected, latest eval_all_ run)
#    - Drive or local: llm_ocr_results/page_X/ folders
#    - Drive or local: logs/ folder
#    - Drive or local: llm_ocr_results/provincia/ folder
#
#  OUTPUTS (local and/or Drive depending on SAVE_MODE):
#    clean_pages/
#      page_X/
#        page_X_semantic.csv     one winning CSV per page
#        page_X_ocr.txt          matching OCR text
#      provincia_semantic.csv    best provincia_di_venezia version
#      provincia_ads.csv         best provincia_ads version
#      selection_log.csv         one row per page, all decisions
#      llm_decisions.jsonl       LLM reasoning for Tier 2 pages
#      statistics_report.md      before/after stats


# CONFIGURATION

DRIVE_ROOT_FOLDER_NAME   = DRIVE_ROOT_FOLDER
DRIVE_OCR_FOLDER_NAME    = DRIVE_OCR_FOLDER
DRIVE_OUTPUT_FOLDER_NAME = DRIVE_CLEAN_PAGES_FOLDER
OUTPUT_DIR               = Path(CLEAN_PAGES_DIR)

# SAVE_MODE lives in config.py 


# Local paths
LOCAL_OCR_ROOT   = Path(OCR_RESULTS_DIR)   
LOCAL_LOGS_DIR   = Path(LOGS_DIR)             
LOCAL_EVAL_DIR   = Path(EVAL_OUTPUT_DIR)  
LOCAL_OUTPUT_DIR = OUTPUT_DIR               

#Settings 
LLM_MODEL       = "gpt-4o"
PAGES           = "all"   # "all" or list of ints e.g. [52, 100, 142]
RUN_MODE = "full"   # "resume" | "full"

# resume = skip pages already in current selection_log.csv
# full   = reprocess everything and overwrite clean outputs
TIER1_SCORE_GAP = 10
CACHE_SCORES = True  # if True, dumps per-page run scores for hyperparameter sweep

DOCUMENT_SECTIONS = {
    "cover_ads":       list(range(1,   14)),
    "index":           list(range(14,  36)),
    "religious":       list(range(36,  38)) + list(range(86, 98)),
    "government":      list(range(38,  86)),
    "professionisti":  list(range(108, 129)),
    "industria":       list(range(132, 319)),
    "indice_generale": list(range(332, 488)),
    "provincia":       list(range(494, 571)),
}

def get_next_version_number(output_dir, base_name, ext):
    # next available N for output_dir/base_name_N.ext
    n = 1
    while (output_dir / f"{base_name}_{n}.{ext}").exists():
        n += 1
    return n


# EVAL AUTO-DETECTION
# Finds the most recent full eval (eval_all_) folder.
# Falls back to most recent partial eval if no full eval exists.
# Falls back to config paths if no eval folder found at all.

def find_latest_eval_files(service, root_id):
    # _results.csv + _page_summary.csv from the latest eval_all_ run, falls back to partial/config paths
    best_local = find_latest_folder_local(LOCAL_EVAL_DIR, "eval_", prefer_prefix="eval_all_")
    if best_local is not None:
        results_files = list(best_local.glob("*_results.csv"))
        summary_files = list(best_local.glob("*_page_summary.csv"))
        if results_files and summary_files:
            if not best_local.name.startswith("eval_all_"):
                print("  WARNING: no full eval found, using most recent partial eval")
            print(f"  Using latest local eval: {best_local.name}")
            return str(results_files[0]), str(summary_files[0]), "local"

    if SAVE_MODE == "local":
        print("  WARNING: no local eval found, using config fallback")
        return EVAL_RESULTS_CSV, EVAL_SUMMARY_CSV, "local"
    # local check done — fall through to Drive if SAVE_MODE allows

    # ── Try Drive ──
    if SAVE_MODE in ("drive", "both") and service and root_id:
        eval_folder_id = find_folder(service, "evaluation", root_id)
        best_drive = find_latest_folder_drive(service, eval_folder_id, "eval_", prefer_prefix="eval_all_")
        if best_drive is not None:
            run_files = list_files_in_folder(service, best_drive["id"])
            results_f = next((f for f in run_files if f["name"].endswith("_results.csv")), None)
            summary_f = next((f for f in run_files if f["name"].endswith("_page_summary.csv")), None)
            if results_f and summary_f:
                if not best_drive["name"].startswith("eval_all_"):
                    print("  WARNING: no full eval found, using most recent partial eval")
                print(f"  Using latest Drive eval: {best_drive['name']}")
                return results_f["id"], summary_f["id"], "drive"

    print("  WARNING: no eval folder found, using config fallback")
    return EVAL_RESULTS_CSV, EVAL_SUMMARY_CSV, "local"


def load_eval_df(results_ref, summary_ref, source_used, service):
    # loads eval DataFrames from a local path or Drive file id
    if source_used == "local":
        eval_df    = pd.read_csv(results_ref)
        summary_df = pd.read_csv(summary_ref)
    else:
        def drive_to_df(file_id):
            text = download_text(service, file_id)
            return pd.read_csv(io.StringIO(text))
        eval_df    = drive_to_df(results_ref)
        summary_df = drive_to_df(summary_ref)
    return eval_df, summary_df


# LOG INDEX
# Loads logs from local logs/ or Drive logs/ depending on SAVE_MODE.

def load_log_index(service, root_id):
    # loads all run_N_scope.log files, local first then Drive
    raw_logs = []  # list of (name, content, created_time)

    if True:  
        if LOCAL_LOGS_DIR.exists():
            local_log_files = list(LOCAL_LOGS_DIR.glob("run_*.log"))
            if local_log_files:
                print(f"  Loading logs from local ({len(local_log_files)} files)")
                for f in local_log_files:
                    content      = f.read_text(encoding="utf-8")
                    created_time = datetime.fromtimestamp(
                        f.stat().st_mtime
                    ).isoformat()
                    raw_logs.append((f.name, content, created_time))
            elif SAVE_MODE == "local":
                print("  WARNING: no local log files found")
                return {}, {}
            # local check done — fall through to Drive if SAVE_MODE allow

    if not raw_logs and SAVE_MODE in ("drive", "both") and service:
        logs_folder = find_folder(service, "logs", root_id)
        if not logs_folder:
            print("  WARNING: no logs/ folder found on Drive")
            return {}, {}
        drive_log_files = [
            f for f in list_files_in_folder(service, logs_folder)
            if f["name"].endswith(".log")
        ]
        for f in drive_log_files:
            content      = download_text(service, f["id"])
            created_time = f.get("createdTime", "")
            raw_logs.append((f["name"], content, created_time))
        print(f"  Loading logs from Drive ({len(drive_log_files)} files)")

    if not raw_logs:
        print("  WARNING: no log files found")
        return {}, {}

    log_index    = {}
    log_metadata = {}

    for name, content, created_time in raw_logs:
        m = re.match(r'^run_(\d+)_(.+)\.log$', name)
        if not m:
            m = re.match(r'^run_(\d{8}_\d{6})_(.+)\.log$', name)
        if not m:
            print(f"  NOTE: {name} did not match log pattern, skipping")
            continue

        run_id  = m.group(1)
        scope   = m.group(2)
        run_num = int(run_id) if run_id.isdigit() else None

        date_m   = re.search(r'(\d{4})(\d{2})(\d{2})', run_id)
        log_date = (
            f"{date_m.group(1)}-{date_m.group(2)}-{date_m.group(3)}"
            if date_m else created_time[:10]
        )

        pages        = {}
        current_page = None
        dates_seen   = set()

        for line in content.splitlines():
            date_in_line = re.search(r'\d{4}-\d{2}-\d{2}', line)
            if date_in_line:
                dates_seen.add(date_in_line.group())
            if "Processing: page_" in line:
                pm = re.search(r'page_(\d+)', line)
                if pm:
                    current_page = int(pm.group(1))
                    pages[current_page] = {"warnings": [], "flags": []}
            if current_page is None:
                continue
            if "WARNING:" in line:
                pages[current_page]["warnings"].append(line.strip())
            if "OCR refusal detected" in line:
                pages[current_page]["flags"].append("ocr_refusal")

        provincia_pages = {p for p in pages if 494 <= p <= 570}
        all_warnings    = sum(len(pages[p]["warnings"]) for p in pages)
        all_refusals    = sum(
            1 for p in pages if "ocr_refusal" in pages[p]["flags"]
        )
        prov_warnings   = sum(
            len(pages[p]["warnings"]) for p in provincia_pages
        )
        prov_refusals   = sum(
            1 for p in provincia_pages
            if "ocr_refusal" in pages[p]["flags"]
        )

        key = (run_num if run_num is not None else run_id, scope)
        log_index[key]    = pages
        log_metadata[key] = {
            "created_time":              created_time,
            "log_date":                  log_date,
            "multi_day":                 len(dates_seen) > 1,
            "dates_seen":                sorted(dates_seen),
            "all_page_count":            len(pages),
            "warning_count_total":       all_warnings,
            "refusal_count_total":       all_refusals,
            "provincia_page_count":      len(provincia_pages),
            "provincia_warning_count":   prov_warnings,
            "provincia_refusal_count":   prov_refusals,
            "filename":                  name,
        }
        print(
            f"  Log: {name} | date={log_date} | "
            f"multi_day={len(dates_seen)>1} | pages={len(pages)} | "
            f"warnings={all_warnings} | refusals={all_refusals}"
        )

    return log_index, log_metadata


def match_file_to_log(file_created_time, log_metadata):
    # log closest before a file's createdTime, or earliest after if none precede
    if not log_metadata or not file_created_time:
        return None

    before = [
        (k, m) for k, m in log_metadata.items()
        if m["created_time"] <= file_created_time
    ]
    after = [
        (k, m) for k, m in log_metadata.items()
        if m["created_time"] > file_created_time
    ]

    if before:
        return max(before, key=lambda x: x[1]["created_time"])[1]
    if after:
        return min(after, key=lambda x: x[1]["created_time"])[1]
    return None


# SCORING — provincia aggregate files

def score_provincia_file(f, eval_row, log_metadata, file_type, csv_text=None):
    # scores a provincia aggregate file 0-100: key field coverage, schema, column diversity, row count, log signals
    score  = 0.0
    issues = []

    rows      = 0.0
    empty_key = 0.0
    schema_ok = True
    prop_flag = False

    if eval_row is not None:
        rows      = float(eval_row.get("Rows") or 0)
        empty_key = float(eval_row.get("Empty Rows") or 0)
        schema_ok = "missing_cols" not in str(eval_row.get("Issues") or "")
        prop_flag = bool(eval_row.get("Propagation Flag"))

    expected_rows = 77 if file_type == "info" else 40

    # Key field coverage
    if rows > 0:
        score += (1 - (empty_key / rows)) * 30
    if prop_flag:
        issues.append("propagation_flag")

    # Schema consistency
    score += 15 if schema_ok else 0

    # Row count
    score += min(rows / expected_rows, 1.0) * 10

    # Column diversity (+ name completeness for ads) from actual CSV content
    if csv_text:
        try:
            rows_parsed = list(csv.DictReader(io.StringIO(csv_text)))
            if rows_parsed:
                header = list(rows_parsed[0].keys())
                col_fill_rates = []
                for col in header:
                    filled = sum(1 for r in rows_parsed if r.get(col, "").strip())
                    col_fill_rates.append(filled / max(len(rows_parsed), 1))
                avg_fill = sum(col_fill_rates) / max(len(col_fill_rates), 1)

                if file_type == "ads":
                    # split: 10 diversity, 5 name completeness
                    score += avg_fill * 10
                    name_col = next(
                        (h for h in header if h.strip().lower() == "name"), None
                    )
                    if name_col:
                        name_vals = [
                            r.get(name_col, "").strip()
                            for r in rows_parsed
                            if r.get(name_col, "").strip()
                        ]
                        if name_vals:
                            complete = sum(
                                1 for n in name_vals if " " in n and len(n) > 5
                            )
                            score += (complete / len(name_vals)) * 5
                else:
                    score += avg_fill * 15
        except Exception:
            score += 7  # neutral score if parsing fails
    else:
        score += 7  # neutral score it if no csv_text provided

    # Log signals matched by creation time
    meta = match_file_to_log(f.get("createdTime", ""), log_metadata)

    if meta:
        pages_covered = meta["all_page_count"]
        prov_covered  = meta["provincia_page_count"]
        all_refusals  = meta["refusal_count_total"]
        all_warnings  = meta["warning_count_total"]
        multi_day     = meta["multi_day"]

        refusal_rate = all_refusals / max(pages_covered, 1)
        warning_rate = all_warnings / max(pages_covered, 1)

        score += max(0, 1 - refusal_rate * 2)   * 10
        score += max(0, 1 - warning_rate * 0.3) *  8
        score += min(prov_covered / 77.0, 1.0)  *  7
        score += 5 if not multi_day else 0

        if all_refusals > 0:
            issues.append(f"ocr_refusals:{all_refusals}")
        if all_warnings > 0:
            issues.append(f"log_warnings:{all_warnings}")
        if multi_day:
            issues.append("multi_day_run")
        if prov_covered < 40:
            issues.append(f"low_provincia_coverage:{prov_covered}/77")
    else:
        score += 15  # neutral score across the 30 log-based points
        issues.append("no_log_matched")

    matched_log_time = meta["created_time"] if meta else ""
    return round(min(score, 100.0), 2), issues, matched_log_time


# TIER 1 AUTO-SELECTION

def tier1_select(scored_runs):
    # auto-selects a winner if only one valid run or the score gap clears TIER1_SCORE_GAP, else None for Tier 2
    valid = [r for r in scored_runs if not r["excluded"]]

    if not valid:
        return None, "all_runs_excluded"
    if len(valid) == 1:
        return valid[0]["run_num"], "only_valid_run"

    valid_sorted = sorted(valid, key=lambda x: x["score"], reverse=True)
    best, second = valid_sorted[0], valid_sorted[1]

    gap = best["score"] - second["score"]
    if gap >= TIER1_SCORE_GAP:
        return (best["run_num"],
                f"score_gap({best['score']:.1f}_vs_{second['score']:.1f})")

    return None, f"ambiguous(best={best['score']:.1f},second={second['score']:.1f})"


# TIER 2 LLM ARBITRATION — per-page CSVs

LLM_PROMPT = """You are helping select the best semantic extraction from a page of a 1947 Venetian almanac.

The page was processed {n_runs} times. Each run produced a CSV.
Pick the one that best captures the actual content: most complete, most accurate
names and addresses, fewest errors. Rows with a Venetian address (sestiere + civic
number) are most valuable for the downstream geocoding task.

OCR TEXT (raw transcription of the page):
---
{ocr_text}
---

{csv_blocks}

Which run is best? Reply ONLY with this JSON:
{{
  "winner": <run number, 1-based integer>,
  "confidence": "high" | "medium" | "low",
  "reason": "<one sentence>"
}}"""


# Formats one run's CSV preview for the LLM-arbitration prompt.
def build_csv_block(run_num, filename, csv_text):
    lines   = [l for l in csv_text.splitlines() if l.strip()]
    preview = lines[:31]
    block   = f"RUN {run_num} ({filename}):\n" + "\n".join(preview)
    if len(lines) > 31:
        block += f"\n... ({len(lines)-31} more rows)"
    return block


def tier2_llm_select(valid_runs, ocr_text, llm):
    # LLM picks the winner among close-scoring runs, falls back to highest score if it fails
    csv_blocks = "\n\n".join(
        build_csv_block(i + 1, r["semantic_file"]["name"], r["csv_text"])
        for i, r in enumerate(valid_runs)
    )
    prompt = LLM_PROMPT.format(
        n_runs    = len(valid_runs),
        ocr_text  = ocr_text[:4000],
        csv_blocks= csv_blocks,
    )
    try:
        res    = llm.invoke([{"role": "user", "content": prompt}])
        raw    = re.sub(r"^```(?:json)?|```$", "",
                        str(res.content).strip()).strip()
        parsed = json.loads(raw)
        idx    = int(parsed["winner"]) - 1
        if 0 <= idx < len(valid_runs):
            return (
                valid_runs[idx]["run_num"],
                parsed.get("confidence", "medium"),
                parsed.get("reason", ""),
            )
        raise ValueError(f"winner index {idx} out of range")
    except Exception as e:
        best = max(valid_runs, key=lambda x: x["score"])
        return best["run_num"], "low", f"llm_failed({e})_fallback_to_score"


# TIER 2 LLM ARBITRATION — provincia aggregate files

def llm_select_provincia(service, scored, file_type, llm):
    # LLM picks the best provincia aggregate version from full CSV + scores + log date
    type_label = (
        "general town information (one row per town, "
        "fields: Provincia, Frazioni, Abitanti, Superficie, Stazione, Prodotti)"
        if file_type == "info"
        else "advertisement entries "
             "(fields: Name, Address, Category, Additional Info, page)"
    )

    for s in scored:
        if s["csv_text"] is None:
            s["csv_text"] = read_file(service, s["file"], SAVE_MODE)
            time.sleep(0.1)

    blocks = []
    for i, s in enumerate(scored):
        lines   = [l for l in s["csv_text"].splitlines() if l.strip()]
        preview = "\n".join(lines)
        blocks.append(
            f"VERSION {i+1} ({s['file']['name']}):\n"
            f"  Score: {s['score']:.1f}/100\n"
            f"  Issues: {s['issues'] or 'none'}\n"
            f"  Matched log date: "
            f"{s['matched_log_time'][:10] if s['matched_log_time'] else 'unknown'}\n"
            f"  Content:\n{preview}"
        )

    prompt = (
        f"You are selecting the best version of a provincia aggregate file "
        f"from a 1947 Venetian almanac pipeline.\n\n"
        f"This file contains {type_label} for towns in the Venice province.\n"
        f"Pick the version that is most complete and accurate.\n\n"
        f"{chr(10).join(blocks)}\n\n"
        f'Reply ONLY with: {{"winner": <1-based int>, '
        f'"confidence": "high"|"medium"|"low", '
        f'"reason": "<one sentence>"}}'
    )

    try:
        res    = llm.invoke([{"role": "user", "content": prompt}])
        raw    = re.sub(r"^```(?:json)?|```$", "",
                        str(res.content).strip()).strip()
        parsed = json.loads(raw)
        idx    = int(parsed["winner"]) - 1
        if 0 <= idx < len(scored):
            return idx, parsed.get("confidence", "medium"), parsed.get("reason", "")
        raise ValueError(f"winner {idx} out of range")
    except Exception as e:
        return 0, "low", f"llm_failed({e})_fallback_to_score"


# PROVINCIA AGGREGATE SELECTION

def select_provincia_aggregates(
    service, root_id, eval_df,
    log_index, log_metadata,
    llm, drive_output_id, output_dir,
    ocr_folder_id=None,
):
    # picks best provincia_di_venezia_N.csv + provincia_ads_N.csv, writes winners flat to output_dir/
    print("\n── Provincia aggregate file selection ──")

    all_files = []

    if SAVE_MODE in ("drive", "both") and service:
        results_id          = ocr_folder_id
        provincia_folder_id = find_folder(service, "provincia", results_id)
        if not provincia_folder_id:
            provincia_folder_id = results_id
            print("  WARNING: no provincia/ subfolder, using llm_ocr_results root")
        else:
            print("  Found Drive provincia/ subfolder")
        all_files = list_files_in_folder(service, provincia_folder_id)

    if not all_files and SAVE_MODE in ("local", "both"):
        local_prov = LOCAL_OCR_ROOT / "provincia"
        if local_prov.exists():
            all_files = [
                {
                    "name":        f.name,
                    "id":          None,
                    "createdTime": datetime.fromtimestamp(
                        f.stat().st_mtime
                    ).isoformat(),
                    "local_path":  str(f)
                }
                for f in local_prov.iterdir()
                if f.is_file() and f.name.endswith(".csv")
            ]
            print(f"  Found {len(all_files)} local provincia files")

    info_files = sorted(
        [f for f in all_files
         if re.match(r'^provincia_di_venezia_\d+\.csv$', f["name"])],
        key=lambda f: (
            int(re.search(r'_(\d+)\.csv$', f["name"]).group(1)),
            f["createdTime"]
        )
    )
    ads_files = sorted(
        [f for f in all_files
         if re.match(r'^provincia_ads_\d+\.csv$', f["name"])],
        key=lambda f: (
            int(re.search(r'_(\d+)\.csv$', f["name"]).group(1)),
            f["createdTime"]
        )
    )

    print(f"  {len(info_files)} info versions, {len(ads_files)} ads versions")

    prov_eval = eval_df[eval_df["Run"] == "PROVINCIA_AGGREGATE"].copy()

    for file_group, output_name, file_type in [
        (info_files, "provincia_semantic.csv", "info"),
        (ads_files,  "provincia_ads.csv",      "ads"),
    ]:
        if not file_group:
            print(f"  No {file_type} files found, skipping")
            continue

        if len(file_group) == 1:
            winner_file = file_group[0]
            tier        = "SINGLE"
            confidence  = "high"
            reason      = "only_one_version"
            print(f"  {file_type}: single version → {winner_file['name']}")
        else:
            scored = []
            for f in file_group:
                fname    = f["name"]
                eval_row = prov_eval[prov_eval["Page"] == fname]
                eval_row = eval_row.iloc[0] if not eval_row.empty else None

                try:
                    csv_text_for_scoring = read_file(service, f, SAVE_MODE)
                except Exception:
                    csv_text_for_scoring = None

                score, iss, matched_time = score_provincia_file(
                    f, eval_row, log_metadata, file_type,
                    csv_text=csv_text_for_scoring
                )
                scored.append({
                    "file":             f,
                    "score":            score,
                    "issues":           iss,
                    "matched_log_time": matched_time,
                    "csv_text":         csv_text_for_scoring,  # reuse in Tier 2
                })

            scored.sort(key=lambda x: x["score"], reverse=True)
            best   = scored[0]
            second = scored[1]
            gap    = best["score"] - second["score"]

            if gap >= TIER1_SCORE_GAP:
                winner_file = best["file"]
                tier        = "TIER1"
                confidence  = "high"
                reason      = (f"score_gap({best['score']:.1f}"
                               f"_vs_{second['score']:.1f})")

            elif gap > 0 and (
                best["matched_log_time"] >= second["matched_log_time"]
            ):
                winner_file = best["file"]
                tier        = "TIER1"
                confidence  = "medium"
                reason      = (f"recency_tiebreaker(gap={gap:.1f},"
                               f"log={best['matched_log_time'][:10]})")
            else:
                winner_idx, confidence, reason = llm_select_provincia(
                    service, scored, file_type, llm
                )
                winner_file = scored[winner_idx]["file"]
                tier        = "TIER2"

            print(f"  {file_type}: [{tier}] → {winner_file['name']} "
                  f"({confidence}) — {reason}")

        content = read_file(service, winner_file, SAVE_MODE)

        if SAVE_MODE in ("local", "both"):
            (output_dir / output_name).write_text(content, encoding="utf-8")

        if SAVE_MODE in ("drive", "both") and drive_output_id:
            upload_text(service, content, output_name, drive_output_id)

        print(f"    → {output_dir / output_name}")


# STATISTICS

def get_section(page_num):
    for name, pages in DOCUMENT_SECTIONS.items():
        if page_num in pages:
            return name
    return "other"


# Before/after stats comparing the raw corpus to the selected-run corpus.
def compute_statistics(semantic_df, compare_df, summary_df, selection_log):
    stats = {}

    sem = semantic_df.drop_duplicates(subset=["Page", "Run"]).copy()
    sem["missing_addr_pct"] = sem["Propagation Note"].apply(extract_missing_pct)
    sem["page_num"] = sem["Page"].apply(
        lambda x: int(re.search(r"(\d+)", str(x)).group(1))
        if re.search(r"(\d+)", str(x)) else 0
    )
    sem["section"] = sem["page_num"].apply(get_section)

    stats["before"] = {
        "total_run_files":           len(sem),
        "pages_covered":             sem["Page"].nunique(),
        "avg_rows_per_run":          round(sem["Rows"].mean(), 1),
        "empty_files":               sem["Issues"].str.contains(
                                         "EMPTY_FILE", na=False).sum(),
        "runs_gt30pct_missing_addr": (sem["missing_addr_pct"] > 30).sum(),
        "runs_100pct_missing_addr":  (sem["missing_addr_pct"] == 100).sum(),
        "rows_per_section":          sem.groupby("section")["Rows"].mean()
                                        .round(1).to_dict(),
        "missing_addr_per_section":  sem.groupby("section")["missing_addr_pct"]
                                        .mean().round(1).to_dict(),
    }

    sims = compare_df["Value Similarity %"].dropna()
    if len(sims) > 0:
        stats["run_agreement"] = {
            "mean_similarity":       round(sims.mean(), 1),
            "median_similarity":     round(sims.median(), 1),
            "pct_above_80":          round(100 * (sims >= 80).sum() / len(sims), 1),
            "pct_below_40":          round(100 * (sims <  40).sum() / len(sims), 1),
            "fully_agree_pairs":     int((sims == 100).sum()),
            "fully_disagree_pairs":  int((sims ==   0).sum()),
        }
    else:
        stats["run_agreement"] = {
            "mean_similarity": 0, "median_similarity": 0,
            "pct_above_80": 0, "pct_below_40": 0,
            "fully_agree_pairs": 0, "fully_disagree_pairs": 0,
        }

    stats["pre_selection_actions"] = summary_df["Action"].value_counts().to_dict()

    if selection_log:
        log_df  = pd.DataFrame(selection_log)
        winners = log_df[log_df["selection_tier"].isin(["SINGLE","TIER1","TIER2"])]

        winner_rows = []
        for _, w in winners.iterrows():
            match = sem[
                (sem["Page"] == w["page"]) &
                (sem["Run"].str.contains(
                    re.escape(w["winner_filename"]), na=False
                ))
            ]
            if not match.empty:
                winner_rows.append(match.iloc[0])

        if winner_rows:
            after_df = pd.DataFrame(winner_rows)
            stats["after"] = {
                "total_winners":                len(after_df),
                "avg_rows_per_winner":          round(after_df["Rows"].mean(), 1),
                "empty_files_in_winners":       after_df["Issues"].str.contains(
                                                    "EMPTY_FILE", na=False).sum(),
                "winners_gt30pct_missing_addr": (
                    after_df["missing_addr_pct"] > 30).sum(),
                "winners_100pct_missing_addr":  (
                    after_df["missing_addr_pct"] == 100).sum(),
                "rows_per_section":             after_df.groupby("section")["Rows"]
                                                    .mean().round(1).to_dict(),
                "missing_addr_per_section":     after_df.groupby("section")
                                                    ["missing_addr_pct"]
                                                    .mean().round(1).to_dict(),
            }

        stats["selection"] = {
            "single_run":  int((log_df["selection_tier"] == "SINGLE").sum()),
            "tier1_auto":  int((log_df["selection_tier"] == "TIER1").sum()),
            "tier2_llm":   int((log_df["selection_tier"] == "TIER2").sum()),
            "failed":      int((log_df["selection_tier"] == "FAILED").sum()),
        }

        tier2 = log_df[log_df["selection_tier"] == "TIER2"]
        if not tier2.empty:
            stats["llm_confidence"] = tier2["confidence"].value_counts().to_dict()
            stats["llm_low_confidence_pages"] = tier2[
                tier2["confidence"] == "low"
            ]["page"].tolist()

        if "naming_bug_detected" in log_df.columns:
            stats["naming_bug_pages"] = log_df[
                log_df["naming_bug_detected"] == True
            ]["page"].tolist()
        else:
            stats["naming_bug_pages"] = []

    return stats


# Writes the statistics report to disk and/or Drive.
def write_statistics_report(stats, output_path):
    lines = [
        "# Venice Almanac — Version Selection Statistics",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        "", "---", "",
        "## 1. Pre-Selection Overview (all runs)", "",
        f"- Total semantic CSV files: **{stats['before']['total_run_files']}**",
        f"- Pages covered: **{stats['before']['pages_covered']}**",
        f"- Average rows per run: **{stats['before']['avg_rows_per_run']}**",
        f"- Empty files: **{stats['before']['empty_files']}**",
        f"- Runs >30% missing address: **{stats['before']['runs_gt30pct_missing_addr']}**",
        f"- Runs 100% missing address: **{stats['before']['runs_100pct_missing_addr']}**",
        "",
        "| Section | Avg Rows | Avg Missing Addr % |",
        "|---------|----------|-------------------|",
    ]
    for section in sorted(stats["before"]["rows_per_section"]):
        rows = stats["before"]["rows_per_section"].get(section, "n/a")
        miss = stats["before"]["missing_addr_per_section"].get(section, "n/a")
        lines.append(f"| {section} | {rows} | {miss}% |")

    lines += [
        "", "---", "",
        "## 2. Run Agreement", "",
        f"- Mean similarity: **{stats['run_agreement']['mean_similarity']}%**",
        f"- Median: **{stats['run_agreement']['median_similarity']}%**",
        f"- Pairs >80%: **{stats['run_agreement']['pct_above_80']}%**",
        f"- Pairs <40%: **{stats['run_agreement']['pct_below_40']}%**",
        "", "---", "",
        "## 3. Pre-Selection Actions", "",
        "| Action | Pages |", "|--------|-------|",
    ]
    for action, count in sorted(
        stats["pre_selection_actions"].items(), key=lambda x: -x[1]
    ):
        lines.append(f"| {action} | {count} |")

    if "selection" in stats:
        lines += [
            "", "---", "",
            "## 4. Selection Results", "",
            f"- Single run: **{stats['selection']['single_run']}**",
            f"- Tier 1 auto: **{stats['selection']['tier1_auto']}**",
            f"- Tier 2 LLM: **{stats['selection']['tier2_llm']}**",
            f"- Failed: **{stats['selection']['failed']}**",
        ]
        if "llm_confidence" in stats:
            lines += [
                "", "### LLM Confidence", "",
                "| Confidence | Pages |", "|------------|-------|",
            ]
            for conf, count in sorted(
                stats["llm_confidence"].items(), key=lambda x: -x[1]
            ):
                lines.append(f"| {conf} | {count} |")
            if stats.get("llm_low_confidence_pages"):
                lines += ["", "**Low confidence pages (manual review):**", ""]
                for p in stats["llm_low_confidence_pages"]:
                    lines.append(f"- {p}")

    if "after" in stats:
        lines += [
            "", "---", "",
            "## 5. Post-Selection Overview", "",
            f"- Total winners: **{stats['after']['total_winners']}**",
            f"- Avg rows per winner: **{stats['after']['avg_rows_per_winner']}**",
            f"- Empty in winners: **{stats['after']['empty_files_in_winners']}**",
            f"- Winners >30% missing: **{stats['after']['winners_gt30pct_missing_addr']}**",
            "",
            "| Section | Avg Rows Before | Avg Rows After | Δ |",
            "|---------|----------------|----------------|---|",
        ]
        for section in sorted(stats["after"]["rows_per_section"]):
            b = stats["before"]["rows_per_section"].get(section, 0)
            a = stats["after"]["rows_per_section"].get(section, 0)
            d = round(float(a) - float(b), 1)
            lines.append(f"| {section} | {b} | {a} | {'+' if d>=0 else ''}{d} |")

    lines += ["", "---", "", "_End of report_"]
    output_path.write_text("\n".join(lines), encoding="utf-8")
    print(f"  Statistics report → {output_path}")


def _generate_stats_text(stats):
    # stats report text without writing to disk, for SAVE_MODE='drive'
    tmp = Path(tempfile.mktemp(suffix=".md"))
    write_statistics_report(stats, tmp)
    text = tmp.read_text(encoding="utf-8")
    tmp.unlink()
    return text


# MAIN

def run_version_selector():
    print("=" * 60)
    print("  VENICE ALMANAC VERSION SELECTOR")
    print("=" * 60)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    log_path     = OUTPUT_DIR / "selection_log.csv"
    llm_log_path = OUTPUT_DIR / "llm_decisions.jsonl"
    version_n  = get_next_version_number(OUTPUT_DIR, "statistics_report", "md")
    stats_path = OUTPUT_DIR / f"statistics_report_{version_n}.md"

    # ── Connect to Drive early if needed ──
    service  = get_drive_service() if SAVE_MODE in ("drive", "both") else None
    root_id  = find_folder(service, DRIVE_ROOT_FOLDER_NAME) if service else None
    if SAVE_MODE in ("drive", "both") and not root_id:
        raise ValueError(f"Drive folder '{DRIVE_ROOT_FOLDER_NAME}' not found")

    # ── Load latest evaluation files ──
    print("\nLocating latest evaluation results ...")
    results_ref, summary_ref, eval_source = find_latest_eval_files(
        service, root_id
    )
    eval_df, summary_df = load_eval_df(
        results_ref, summary_ref, eval_source, service
    )
    semantic_df = eval_df[
        eval_df["Run"].str.contains("semantic", na=False)
    ].copy()
    compare_df  = eval_df[
        eval_df["Run"].str.startswith("compare_", na=False)
    ].copy()
    print(f"  {len(semantic_df)} semantic rows | "
          f"{summary_df['Page'].nunique()} pages in summary")

    # ── Resume support ──
    completed_pages = set()
    log_open_mode   = "w"  # default for "full" or first-ever run

    if RUN_MODE == "resume" and log_path.exists():
        existing_log = pd.read_csv(log_path)
        completed_pages = set(
            re.sub(r'\.[a-zA-Z0-9]+$', '', str(p))
            for p in existing_log["page"].tolist()
        )
        log_open_mode = "a"
        print(f"Resume mode: {len(completed_pages)} pages already done")

    elif RUN_MODE == "full":
        # Reprocess everything, ignore any prior log, start fresh
        completed_pages = set()
        log_open_mode = "w"
        print("Full mode: reprocessing all pages (fresh selection_log)")

    # ── Drive output folder ──
    drive_output_id = None
    if SAVE_MODE in ("drive", "both"):
        ocr_folder = find_folder(service, DRIVE_OCR_FOLDER_NAME, root_id)
        if not ocr_folder:
            raise ValueError(
                f"'{DRIVE_OCR_FOLDER_NAME}' not found under root"
            )
        drive_output_id = get_or_create_folder(
            service, DRIVE_OUTPUT_FOLDER_NAME, root_id
        )
    else:
        ocr_folder = None

    # ── LLM setup ──
    print("Loading LLM ...")
    with open(Path(OPENAI_KEY_FILE)) as f:
        params = dict(v.strip().split("=", 1) for v in f if "=" in v)
    llm = ChatOpenAI(model=LLM_MODEL, openai_api_key=params["api_key"])

    # ── Load logs ──
    print("\nLoading log index ...")
    log_index, log_metadata = load_log_index(service, root_id)

    # ── List page folders ──
    print("\nListing page folders ...")
    page_folders = []

    if SAVE_MODE in ("drive", "both") and ocr_folder:
        page_folders = [
            f for f in list_files_in_folder(service, ocr_folder)
            if f["mimeType"] == "application/vnd.google-apps.folder"
            and re.match(r'^page_\d+$', f["name"])
        ]

    if not page_folders and SAVE_MODE in ("local", "both"):
        page_folders = [
            {
                "name":        d.name,
                "id":          None,
                "createdTime": "",
                "local_path":  d
            }
            for d in sorted(LOCAL_OCR_ROOT.iterdir())
            if d.is_dir() and re.match(r'^page_\d+$', d.name)
        ]

    # Filter and sort
    if PAGES != "all":
        page_nums = set(PAGES)
        page_folders = [
            f for f in page_folders
            if (m := re.search(r"(\d+)", f["name"]))
            and int(m.group(1)) in page_nums
        ]
    page_folders.sort(
        key=lambda f: int(re.search(r"(\d+)", f["name"]).group(1))
        if re.search(r"(\d+)", f["name"]) else 9999
    )
    print(f"  {len(page_folders)} page folders")

    # ── Open log files ──
    log_file     = open(log_path, log_open_mode, newline="", encoding="utf-8")
    log_fields   = [
        "page", "page_num", "n_runs_found", "n_valid_runs",
        "winner_filename", "selection_tier", "confidence",
        "winner_score", "reason", "all_scores",
        "ocr_file_found", "timestamp"
    ]
    log_writer   = csv.DictWriter(log_file, fieldnames=log_fields)
    if log_open_mode == "w":
        log_writer.writeheader()
    llm_log_file = open(llm_log_path, log_open_mode, encoding="utf-8")

    stats_counters = {"tier1": 0, "tier2": 0, "single": 0,
                      "skipped": 0, "failed": 0}
    selection_log  = []
    scores_cache   = []

    for folder in page_folders:
        page_name = re.sub(r'\.[a-zA-Z0-9]+$', '', folder["name"])
        m         = re.search(r"(\d+)", page_name)
        page_num  = int(m.group(1)) if m else 0

        if page_name in completed_pages:
            stats_counters["skipped"] += 1
            continue

        print(f"  [{page_num:>4}] {page_name}", end=" ... ", flush=True)

        # List files in this page folder
        if SAVE_MODE in ("drive", "both") and folder.get("id"):
            all_files = list_files_in_folder(service, folder["id"])
        else:
            local_page_path = (
                folder.get("local_path") or LOCAL_OCR_ROOT / page_name
            )
            all_files = [
                {
                    "name":        f.name,
                    "id":          None,
                    "createdTime": datetime.fromtimestamp(
                        f.stat().st_mtime
                    ).isoformat(),
                    "local_path":  str(f)
                }
                for f in Path(local_page_path).iterdir()
                if f.is_file()
            ]

        semantic_files = [
            f for f in all_files
            if "_semantic_" in f["name"] and f["name"].endswith(".csv")
        ]
        ocr_files = [
            f for f in all_files
            if "_ocr_" in f["name"] and f["name"].endswith(".txt")
        ]

        if not semantic_files:
            print("NO SEMANTIC FILES — skip")
            stats_counters["failed"] += 1
            log_writer.writerow({
                "page": page_name, "page_num": page_num,
                "n_runs_found": 0, "n_valid_runs": 0,
                "winner_filename": "", "selection_tier": "FAILED",
                "confidence": "", "winner_score": "",
                "reason": "no_semantic_files_found",
                "all_scores": "", "ocr_file_found": False,
                "timestamp": datetime.now().isoformat()
            })
            continue

        paired = pair_runs(semantic_files, ocr_files)

        page_eval = semantic_df[
            semantic_df["Page"] == page_name
        ].drop_duplicates(subset="Run").reset_index(drop=True)

        avg_sim = compute_avg_value_sim(page_name, compare_df)

        scored_runs = []
        for run in paired:
            n        = run["run_num"]
            eval_row = None
            for _, row in page_eval.iterrows():
                rn = re.search(r'_(\d+)\.csv$', str(row.get("Run", "")))
                if rn and int(rn.group(1)) == n:
                    eval_row = row
                    break

            excl, excl_reason = is_excluded(eval_row)
            rows  = float(eval_row.get("Rows") or 0) if eval_row is not None else 0

            # Load CSV for richer scoring (column diversity, name completeness)
            # Only for non-excluded runs to avoid unnecessary reads
            csv_text_for_scoring = None
            if not excl:
                try:
                    csv_text_for_scoring = read_file(service, run["semantic_file"], SAVE_MODE)
                except Exception:
                    pass

            score = score_run(eval_row, avg_sim, csv_text=csv_text_for_scoring)

            scored_runs.append({
                "run_num":          n,
                "semantic_file":    run["semantic_file"],
                "ocr_file":         run["ocr_file"],
                "excluded":         excl,
                "exclusion_reason": excl_reason,
                "rows":             rows,
                "score":            score,
                "csv_text":         csv_text_for_scoring,  # reuse in Tier 2 if needed
            })

        if CACHE_SCORES:
            scores_cache.append({
                "page":     page_name,
                "page_num": page_num,
                "run_nums": [r["run_num"] for r in scored_runs],
                "scores":   [r["score"] for r in scored_runs],
                "excluded": [r["excluded"] for r in scored_runs],
            })

        n_valid = sum(1 for r in scored_runs if not r["excluded"])

        # ── Selection ──
        if len(scored_runs) == 1:
            winner                   = scored_runs[0]
            tier, confidence, reason = "SINGLE", "high", "only_one_run"
            stats_counters["single"] += 1

        else:
            winner_run_num, t1_reason = tier1_select(scored_runs)

            if winner_run_num is not None:
                winner = next(
                    r for r in scored_runs if r["run_num"] == winner_run_num
                )
                tier, confidence, reason = "TIER1", "high", t1_reason
                stats_counters["tier1"] += 1

            else:
                valid_runs = [r for r in scored_runs if not r["excluded"]]

                if not valid_runs:
                    winner = max(scored_runs, key=lambda x: x["rows"])
                    tier, confidence, reason = (
                        "FAILED", "low", "all_runs_excluded_picked_most_rows"
                    )
                    stats_counters["failed"] += 1

                else:
                    for r in valid_runs:
                        # Reuse CSV already loaded during scoring if available
                        if r["csv_text"] is None:
                            r["csv_text"] = read_file(service, r["semantic_file"], SAVE_MODE)
                            time.sleep(0.1)

                    ocr_text = ""
                    if valid_runs[0]["ocr_file"]:
                        ocr_text = read_file(service, valid_runs[0]["ocr_file"], SAVE_MODE) or ""

                    winner_run_num, confidence, reason = tier2_llm_select(
                        valid_runs, ocr_text, llm
                    )
                    winner = next(
                        r for r in valid_runs if r["run_num"] == winner_run_num
                    )
                    tier = "TIER2"
                    stats_counters["tier2"] += 1

                    llm_log_file.write(json.dumps({
                        "page":            page_name,
                        "page_num":        page_num,
                        "winner":          winner["semantic_file"]["name"],
                        "confidence":      confidence,
                        "reason":          reason,
                        "runs_considered": [r["semantic_file"]["name"]
                                            for r in valid_runs],
                        "scores":          [r["score"] for r in valid_runs],
                        "timestamp":       datetime.now().isoformat()
                    }) + "\n")

        if winner["csv_text"] is None:
            winner["csv_text"] = read_file(service, winner["semantic_file"], SAVE_MODE)
        if winner["csv_text"] is None:
            print("ERROR: could not read winner CSV — skipping page")
            stats_counters["failed"] += 1
            continue

        ocr_text       = ""
        ocr_file_found = False
        if winner["ocr_file"]:
            ocr_text = read_file(service, winner["ocr_file"], SAVE_MODE) or ""
            ocr_file_found = bool(ocr_text)

        clean_csv_name = f"page_{page_num}_semantic.csv"
        clean_ocr_name = f"page_{page_num}_ocr.txt"
        page_out_dir   = OUTPUT_DIR / f"page_{page_num}"

        if SAVE_MODE in ("local", "both"):
            page_out_dir.mkdir(parents=True, exist_ok=True)
            (page_out_dir / clean_csv_name).write_text(
                winner["csv_text"], encoding="utf-8"
            )
            if ocr_text:
                (page_out_dir / clean_ocr_name).write_text(
                    ocr_text, encoding="utf-8"
                )

        if SAVE_MODE in ("drive", "both") and drive_output_id:
            page_drive_folder = get_or_create_folder(
                service, f"page_{page_num}", drive_output_id
            )
            upload_text(service, winner["csv_text"],
                        clean_csv_name, page_drive_folder)
            if ocr_text:
                upload_text(service, ocr_text,
                            clean_ocr_name, page_drive_folder)

        all_scores = "; ".join(
            f"{r['semantic_file']['name']}={r['score']:.1f}"
            for r in scored_runs
        )
        log_entry = {
            "page":            page_name,
            "page_num":        page_num,
            "n_runs_found":    len(scored_runs),
            "n_valid_runs":    n_valid,
            "winner_filename": winner["semantic_file"]["name"],
            "selection_tier":  tier,
            "confidence":      confidence,
            "winner_score":    winner["score"],
            "reason":          reason,
            "all_scores":      all_scores,
            "ocr_file_found":  ocr_file_found,
            "timestamp":       datetime.now().isoformat()
        }
        log_writer.writerow(log_entry)
        log_file.flush()
        selection_log.append(log_entry)

        print(f"[{tier}] → page_{page_num}/ ({confidence})")

    log_file.close()
    llm_log_file.close()

    # Upload selection log and LLM decisions log to Drive
    if SAVE_MODE in ("drive", "both") and drive_output_id:
        upload_text(
            service,
            log_path.read_text(encoding="utf-8"),
            log_path.name,
            drive_output_id
        )
        if llm_log_path.exists() and llm_log_path.stat().st_size > 0:
            upload_text(
                service,
                llm_log_path.read_text(encoding="utf-8"),
                llm_log_path.name,
                drive_output_id
            )

    # ── Provincia aggregate selection ──
    select_provincia_aggregates(
        service         = service,
        root_id         = root_id,
        eval_df         = eval_df,
        log_index       = log_index,
        log_metadata    = log_metadata,
        llm             = llm,
        drive_output_id = drive_output_id,
        output_dir      = OUTPUT_DIR,
        ocr_folder_id   = ocr_folder,
    )

    # ── Statistics ──
    if CACHE_SCORES and scores_cache:
        cache_path = OUTPUT_DIR / f"scores_cache_{version_n}.json"
        cache_path.write_text(json.dumps(scores_cache, indent=2), encoding="utf-8")
        print(f"  Scores cache → {cache_path}")
        if SAVE_MODE in ("drive", "both") and drive_output_id:
            upload_text(
                service, cache_path.read_text(encoding="utf-8"),
                cache_path.name, drive_output_id
            )
    print("\nComputing statistics ...")
    stats = compute_statistics(
        semantic_df, compare_df, summary_df, selection_log
    )

    if SAVE_MODE in ("local", "both"):
        write_statistics_report(stats, stats_path)

    if SAVE_MODE in ("drive", "both") and drive_output_id:
        stats_content = (
            stats_path.read_text(encoding="utf-8")
            if stats_path.exists()
            else _generate_stats_text(stats)
        )
        upload_text(service, stats_content, stats_path.name, drive_output_id)

    # Snapshot the current selection log and LLM decisions for this run, using the same version number as the stats report.
    if log_path.exists():
        snapshot_log = OUTPUT_DIR / f"selection_log_{version_n}.csv"
        shutil.copy(log_path, snapshot_log)
        if SAVE_MODE in ("drive", "both") and drive_output_id:
            upload_text(
                service, snapshot_log.read_text(encoding="utf-8"),
                snapshot_log.name, drive_output_id
            )
    if llm_log_path.exists() and llm_log_path.stat().st_size > 0:
        snapshot_llm = OUTPUT_DIR / f"llm_decisions_{version_n}.jsonl"
        shutil.copy(llm_log_path, snapshot_llm)
        if SAVE_MODE in ("drive", "both") and drive_output_id:
            upload_text(
                service, snapshot_llm.read_text(encoding="utf-8"),
                snapshot_llm.name, drive_output_id
            )

    print("\n" + "=" * 60)
    print("  DONE")
    print("=" * 60)
    print(f"  Single:   {stats_counters['single']}")
    print(f"  Tier 1:   {stats_counters['tier1']}")
    print(f"  Tier 2:   {stats_counters['tier2']}")
    print(f"  Failed:   {stats_counters['failed']}")
    print(f"  Skipped:  {stats_counters['skipped']}")
    print(f"\n  Output:  {OUTPUT_DIR}/")
    print(f"  Log:     {log_path}")
    print(f"  LLM log: {llm_log_path}")
    print(f"  Stats:   {stats_path}")

Run it


In [2]:
run_version_selector()

  VENICE ALMANAC VERSION SELECTOR

Locating latest evaluation results ...
  Using latest Drive eval: eval_all_20260612_1130
  1537 semantic rows | 564 pages in summary
  Resuming: 571 pages already done
Loading LLM ...

Loading log index ...
  Loading logs from Drive (3 files)
  Log: run_1_all_may4.log | date=2026-05-10 | multi_day=False | pages=575 | warnings=375 | refusals=21
  Log: run_3_all_may7.log | date=2026-05-11 | multi_day=False | pages=575 | warnings=391 | refusals=26
  Log: run_4_range_494_570_may13.log | date=2026-05-15 | multi_day=False | pages=77 | warnings=0 | refusals=2

Listing page folders ...
  571 page folders

── Provincia aggregate file selection ──
  Found Drive provincia/ subfolder
  3 info versions, 3 ads versions
  info: [TIER2] → provincia_di_venezia_1.csv (high) — Version 1 has the highest score and offers the most complete set of data with consistent formatting and detail.
    → /Users/charlottegarcia/Desktop/thesis/Venezia_Almanac_1947/clean_pages/provinc

## Revised design — score-only 
Self-contained rewrite, no LLM arbitration tier. This is what the pipeline actually runs.


In [ ]:
# SCORE-ONLY VERSION SELECTOR — FULLY SELF-CONTAINED

# ── Configuration 
SAVE_MODE_SCORE              = SAVE_MODE   # in config.SAVE_MODE
PAGES_SCORE                  = list(range(494, 571)) + [41, 130, 161, 170, 320, 367, 383, 458]    # "all" or list of ints e.g. [1, 2, 3]
READ_CSV_FOR_SCORING         = True      # True = 20 pts more precision, ~30 min extra
TIED_THRESHOLD               = 0.1       # scores within this gap = tied → recency wins
EVAL_RUN_NAME = None  
# Set to a specific folder name to force-load that eval, e.g.: EVAL_RUN_NAME = "eval_pages_494_570_20260629_1430"
# Leave as None to auto-detect latest eval_all_


# ── Resume / partial run control ─────────────────────────────────────
RUN_MODE_SCORE = "full"     # "full"   = reprocess everything from scratch
                             # "resume" = skip pages already in score_selection_log.csv


DRIVE_ROOT_FOLDER_NAME       = DRIVE_ROOT_FOLDER
DRIVE_OCR_FOLDER_NAME        = DRIVE_OCR_FOLDER
DRIVE_OUTPUT_SCORE_FOLDER    = DRIVE_CLEAN_PAGES_FOLDER   # "clean_pages"

LLM_LOG_DRIVE_FOLDER         = "clean_pages_llm"
LLM_LOG_FILENAME             = "selection_log_1.csv"

LOCAL_OCR_ROOT               = Path(OCR_RESULTS_DIR)
LOCAL_EVAL_DIR               = Path(EVAL_OUTPUT_DIR)
LOCAL_OUTPUT_SCORE_DIR       = Path(CLEAN_PAGES_DIR)

DOCUMENT_SECTIONS = {
    "cover_ads":       list(range(1,   14)),
    "index":           list(range(14,  36)),
    "religious":       list(range(36,  38)) + list(range(86, 98)),
    "government":      list(range(38,  86)),
    "professionisti":  list(range(108, 129)),
    "industria":       list(range(132, 319)),
    "indice_generale": list(range(332, 488)),
    "provincia":       list(range(494, 571)),
}

print("=" * 60)
print("  SCORE-ONLY VERSION SELECTOR")
print("=" * 60)
print(f"  SAVE_MODE_SCORE      : {SAVE_MODE_SCORE}")
print(f"  READ_CSV_FOR_SCORING : {READ_CSV_FOR_SCORING}")
print(f"  Output folder        : {DRIVE_OUTPUT_SCORE_FOLDER}")

# EVAL LOADING

def find_latest_eval_local():
    best = find_latest_folder_local(LOCAL_EVAL_DIR, "eval_", prefer_prefix="eval_all_")
    if best:
        results = list(best.glob("*_results.csv"))
        summary = list(best.glob("*_page_summary.csv"))
        if results and summary:
            return str(results[0]), str(summary[0])
    return None, None


# Finds the most recent evaluation folder on Drive.
def find_latest_eval_drive(service, root_id):
    eval_id = find_folder(service, "evaluation", root_id)
    best = find_latest_folder_drive(service, eval_id, "eval_", prefer_prefix="eval_all_")
    if best is None:
        return None, None
    run_files = list_files_in_folder(service, best["id"])
    results_f = next((f for f in run_files if f["name"].endswith("_results.csv")), None)
    summary_f = next((f for f in run_files if f["name"].endswith("_page_summary.csv")), None)
    if results_f and summary_f:
        return results_f["id"], summary_f["id"]
    return None, None


# DRIVE SETUP

score_service  = get_drive_service() if SAVE_MODE_SCORE in ("drive", "both") else None
score_root_id  = find_folder(score_service, DRIVE_ROOT_FOLDER_NAME) if score_service else None

if SAVE_MODE_SCORE in ("drive", "both") and not score_root_id:
    raise ValueError(f"Drive root folder '{DRIVE_ROOT_FOLDER_NAME}' not found")

score_drive_output_id = None
if SAVE_MODE_SCORE in ("drive", "both"):
    score_drive_output_id = get_or_create_folder(
        score_service, DRIVE_OUTPUT_SCORE_FOLDER, score_root_id
    )
    print(f"  Drive output folder: {DRIVE_OUTPUT_SCORE_FOLDER} ✓")

if SAVE_MODE_SCORE in ("local", "both"):
    LOCAL_OUTPUT_SCORE_DIR.mkdir(parents=True, exist_ok=True)

print("\nLoading eval data ...")
eval_df = summary_df = semantic_df = compare_df = None

# Finds a named evaluation folder on Drive.
def find_specific_eval_drive(service, root_id, run_name):
    eval_id = find_folder(service, "evaluation", root_id)
    if not eval_id:
        return None, None
    run_id = find_folder(service, run_name, eval_id)
    if not run_id:
        return None, None
    run_files = list_files_in_folder(service, run_id)
    results_f = next((f for f in run_files if f["name"].endswith("_results.csv")), None)
    summary_f = next((f for f in run_files if f["name"].endswith("_page_summary.csv")), None)
    if results_f and summary_f:
        return results_f["id"], summary_f["id"]
    return None, None

# Try local first
results_ref, summary_ref = find_latest_eval_local()
if results_ref:
    eval_df    = pd.read_csv(results_ref)
    summary_df = pd.read_csv(summary_ref)
    print(f"  Eval loaded locally")
elif SAVE_MODE_SCORE in ("drive", "both") and score_service:
    if EVAL_RUN_NAME:
        results_ref, summary_ref = find_specific_eval_drive(
            score_service, score_root_id, EVAL_RUN_NAME
        )
        print(f"  Loading specific eval: {EVAL_RUN_NAME}")
    else:
        results_ref, summary_ref = find_latest_eval_drive(
            score_service, score_root_id
        )
    if results_ref:
        eval_df    = pd.read_csv(io.StringIO(download_text(score_service, results_ref)))
        summary_df = pd.read_csv(io.StringIO(download_text(score_service, summary_ref)))
        print(f"  Eval loaded from Drive")

if eval_df is None:
    raise RuntimeError("Could not load eval data. Run the evaluation notebook first.")

semantic_df = eval_df[eval_df["Run"].str.contains("semantic", na=False)].copy()
compare_df  = eval_df[eval_df["Run"].str.startswith("compare_", na=False)].copy()
print(f"  {len(semantic_df)} semantic rows | {summary_df['Page'].nunique()} pages")

# LOAD LLM SELECTION LOG FOR COMPARISON

print("\nLoading LLM selection log ...")
llm_log_df = None

if SAVE_MODE_SCORE in ("drive", "both") and score_service:
    llm_folder_id = find_folder(score_service, LLM_LOG_DRIVE_FOLDER, score_root_id)
    if llm_folder_id:
        llm_files = list_files_in_folder(score_service, llm_folder_id)
        llm_file  = next(
            (f for f in llm_files if f["name"] == LLM_LOG_FILENAME), None
        )
        if llm_file:
            llm_log_df = pd.read_csv(
                io.StringIO(download_text(score_service, llm_file["id"]))
            )
            print(f"  LLM log loaded from Drive: {len(llm_log_df)} rows")
        else:
            print(f"  WARNING: {LLM_LOG_FILENAME} not found in {LLM_LOG_DRIVE_FOLDER}")
    else:
        print(f"  WARNING: folder {LLM_LOG_DRIVE_FOLDER} not found on Drive")

if llm_log_df is None:
    print("  LLM log not found — comparison will show unknown for LLM winners")
    llm_log_df = pd.DataFrame(columns=["page", "winner_filename", "winner_score"])

llm_lookup = {
    str(row.get("page", "")).strip(): {
        "llm_winner":        str(row.get("winner_filename", "") or ""),
        "llm_winner_score":  row.get("winner_score", None),
    }
    for _, row in llm_log_df.iterrows()
    if str(row.get("page", "")).strip()
}

# LIST PAGE FOLDERS

print("\nListing page folders ...")
score_page_folders = []

if SAVE_MODE_SCORE in ("drive", "both") and score_service:
    ocr_folder_id = find_folder(score_service, DRIVE_OCR_FOLDER_NAME, score_root_id)
    if not ocr_folder_id:
        raise ValueError(f"OCR folder '{DRIVE_OCR_FOLDER_NAME}' not found on Drive")
    score_page_folders = [
        f for f in list_files_in_folder(score_service, ocr_folder_id)
        if f["mimeType"] == "application/vnd.google-apps.folder"
        and re.match(r'^page_\d+$', f["name"])
    ]

if not score_page_folders and SAVE_MODE_SCORE in ("local", "both"):
    score_page_folders = [
        {
            "name":        d.name,
            "id":          None,
            "createdTime": "",
            "local_path":  d,
        }
        for d in sorted(LOCAL_OCR_ROOT.iterdir())
        if d.is_dir() and re.match(r'^page_\d+$', d.name)
    ]

if PAGES_SCORE != "all":
    page_nums_filter = set(PAGES_SCORE)
    score_page_folders = [
        f for f in score_page_folders
        if (m := re.search(r"(\d+)", f["name"])) and int(m.group(1)) in page_nums_filter
    ]

score_page_folders.sort(
    key=lambda f: int(re.search(r"(\d+)", f["name"]).group(1))
    if re.search(r"(\d+)", f["name"]) else 9999
)
print(f"  {len(score_page_folders)} page folders found")


# ── Load existing log if resuming ────────────────────────────────────────
# ── Initialise log accumulators ──────────────────────────────────────────
score_log_rows  = []
comparison_rows = []
completed_pages = set()
log_open_mode   = "w"

if RUN_MODE_SCORE == "resume":
    # Try to load existing score_selection_log from Drive or local
    existing_log_df = None

    if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
        existing_log_files = list_files_in_folder(
            score_service, score_drive_output_id
        )
        existing_log_file = next(
            (f for f in existing_log_files
             if f["name"] == "score_selection_log.csv"),
            None
        )
        if existing_log_file:
            try:
                text = download_text(score_service, existing_log_file["id"])
                existing_log_df = pd.read_csv(io.StringIO(text))
                print(f"  Resume: loaded existing log from Drive "
                      f"({len(existing_log_df)} rows)")
            except Exception as e:
                print(f"  WARNING: could not load existing log — {e}")

    if existing_log_df is None and SAVE_MODE_SCORE in ("local", "both"):
        local_log = LOCAL_OUTPUT_SCORE_DIR / "score_selection_log.csv"
        if local_log.exists():
            existing_log_df = pd.read_csv(local_log)
            print(f"  Resume: loaded existing log locally "
                  f"({len(existing_log_df)} rows)")

    if existing_log_df is not None:
        completed_pages = set(existing_log_df["page"].astype(str).tolist())
        # Pre-populate log rows and comparison rows from existing data
        score_log_rows  = existing_log_df.to_dict("records")
        # Try to load existing comparison too
        if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
            existing_comp_file = next(
                (f for f in existing_log_files
                 if f["name"] == "score_vs_llm_comparison.csv"),
                None
            )
            if existing_comp_file:
                try:
                    text2 = download_text(score_service, existing_comp_file["id"])
                    existing_comp_df = pd.read_csv(io.StringIO(text2))
                    comparison_rows  = existing_comp_df.to_dict("records")
                except Exception:
                    comparison_rows = []
        print(f"  Resume: {len(completed_pages)} pages already done — skipping")
    else:
        print("  Resume requested but no existing log found — starting fresh")

    log_open_mode = "a"
else:
    log_open_mode = "w"
    completed_pages = set()

# MAIN SELECTION LOOP

print("\nRunning score-only selection ...")

# Load existing log rows to preserve entries for pages not in this run
_existing_log_rows  = []
_existing_comp_rows = []

if PAGES_SCORE != "all":
    # Partial run — preserve existing log entries for pages outside our scope
    _pages_in_scope = set(PAGES_SCORE) if isinstance(PAGES_SCORE, list) else set()
    
    if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
        _existing_files = list_files_in_folder(score_service, score_drive_output_id)
        _log_f = next((f for f in _existing_files
                       if f["name"] == "score_selection_log.csv"), None)
        _cmp_f = next((f for f in _existing_files
                       if f["name"] == "score_vs_llm_comparison.csv"), None)
        if _log_f:
            try:
                _df = pd.read_csv(io.StringIO(download_text(score_service, _log_f["id"])))
                _existing_log_rows = [
                    r for r in _df.to_dict("records")
                    if int(r.get("page_num", 0)) not in _pages_in_scope
                ]
                print(f"  Preserving {len(_existing_log_rows)} existing log rows "
                      f"outside this run's scope")
            except Exception as e:
                print(f"  WARNING: could not load existing log for merge — {e}")
        if _cmp_f:
            try:
                _df2 = pd.read_csv(io.StringIO(download_text(score_service, _cmp_f["id"])))
                _existing_comp_rows = [
                    r for r in _df2.to_dict("records")
                    if int(r.get("page_num", 0)) not in _pages_in_scope
                ]
            except Exception:
                pass
    elif SAVE_MODE_SCORE in ("local", "both"):
        _local_log = LOCAL_OUTPUT_SCORE_DIR / "score_selection_log.csv"
        _local_cmp = LOCAL_OUTPUT_SCORE_DIR / "score_vs_llm_comparison.csv"
        if _local_log.exists():
            try:
                _df = pd.read_csv(_local_log)
                _existing_log_rows = [
                    r for r in _df.to_dict("records")
                    if int(r.get("page_num", 0)) not in _pages_in_scope
                ]
                print(f"  Preserving {len(_existing_log_rows)} existing log rows")
            except Exception:
                pass
        if _local_cmp.exists():
            try:
                _df2 = pd.read_csv(_local_cmp)
                _existing_comp_rows = [
                    r for r in _df2.to_dict("records")
                    if int(r.get("page_num", 0)) not in _pages_in_scope
                ]
            except Exception:
                pass

score_log_rows  = []   # new rows from this run only — merged with existing at save time
comparison_rows = []
counters        = {
    "single": 0, "score_clear": 0, "score_tied": 0,
    "excluded_fallback": 0, "failed": 0
}
t_start = time.time()

for folder in score_page_folders:
    page_name = re.sub(r'\.[a-zA-Z0-9]+$', '', folder["name"])
    m         = re.search(r"(\d+)", page_name)
    page_num  = int(m.group(1)) if m else 0

    # Skip if already done in resume mode
    if page_name in completed_pages:
        continue

    # List files in page folder
    if SAVE_MODE_SCORE in ("drive", "both") and folder.get("id"):
        all_files = list_files_in_folder(score_service, folder["id"])
    else:
        local_page_path = folder.get("local_path") or LOCAL_OCR_ROOT / page_name
        all_files = [
            {
                "name":        f.name,
                "id":          None,
                "createdTime": datetime.fromtimestamp(f.stat().st_mtime).isoformat(),
                "local_path":  str(f),
            }
            for f in Path(local_page_path).iterdir()
            if f.is_file()
        ]

    semantic_files = [
        f for f in all_files
        if "_semantic_" in f["name"] and f["name"].endswith(".csv")
    ]
    ocr_files = [
        f for f in all_files
        if "_ocr_" in f["name"] and f["name"].endswith(".txt")
    ]

    if not semantic_files:
        print(f"  [{page_num:>4}] NO SEMANTIC FILES — skip")
        counters["failed"] += 1
        score_log_rows.append({
            "page":             page_name,
            "page_num":         page_num,
            "n_runs_found":     0,
            "n_valid_runs":     0,
            "winner_filename":  "",
            "selection_method": "FAILED",
            "winner_score":     None,
            "scores_tied":      False,
            "reason":           "no_semantic_files_found",
            "all_scores":       "",
            "single_run_flag":  False,
            "ocr_file_found":   False,
            "timestamp":        datetime.now().isoformat(),
        })
        continue

    paired   = pair_runs(semantic_files, ocr_files)
    page_eval = semantic_df[
        semantic_df["Page"] == page_name
    ].drop_duplicates(subset="Run").reset_index(drop=True)
    avg_sim  = compute_avg_value_sim(page_name, compare_df)

    # Score each run
    scored_runs = []
    for run in paired:
        n        = run["run_num"]
        eval_row = None
        for _, erow in page_eval.iterrows():
            rn = re.search(r'_(\d+)\.csv$', str(erow.get("Run", "")))
            if rn and int(rn.group(1)) == n:
                eval_row = erow
                break

        excl, excl_reason = is_excluded(eval_row)

        csv_text_for_scoring = None
        if READ_CSV_FOR_SCORING and not excl:
            try:
                csv_text_for_scoring = read_file(score_service, run["semantic_file"], SAVE_MODE_SCORE)
            except Exception:
                pass

        score = score_run(eval_row, avg_sim, csv_text=csv_text_for_scoring)

        scored_runs.append({
            "run_num":          n,
            "semantic_file":    run["semantic_file"],
            "ocr_file":         run["ocr_file"],
            "excluded":         excl,
            "exclusion_reason": excl_reason,
            "score":            score,
            "created_time":     run["semantic_file"].get("createdTime", ""),
            "csv_text":         csv_text_for_scoring,
        })

    all_scores_str = "; ".join(
        f"{r['semantic_file']['name']}={r['score']:.2f}"
        for r in scored_runs
    )
    n_valid        = sum(1 for r in scored_runs if not r["excluded"])
    single_run_flag = len(scored_runs) == 1

    # ── Selection ────────────────────────────────────────────────────
    if single_run_flag:
        winner           = scored_runs[0]
        selection_method = "SINGLE"
        scores_tied      = False
        reason           = "only_one_run"
        if single_run_flag:
            print(f"  [{page_num:>4}] FLAG: only one run found (should not happen)")
        counters["single"] += 1

    else:
        valid_runs = [r for r in scored_runs if not r["excluded"]]

        if not valid_runs:
            winner           = max(scored_runs, key=lambda x: x["score"])
            selection_method = "EXCLUDED_FALLBACK"
            scores_tied      = False
            reason           = "all_runs_excluded_picked_highest_score"
            counters["excluded_fallback"] += 1
            print(f"  [{page_num:>4}] WARNING: all runs excluded — "
                  f"fallback: {winner['semantic_file']['name']}")
        else:
            valid_sorted = sorted(
                valid_runs,
                key=lambda x: (x["score"], x["created_time"]),
                reverse=True,
            )
            best        = valid_sorted[0]
            second      = valid_sorted[1] if len(valid_sorted) > 1 else None
            score_gap   = (best["score"] - second["score"]) if second else 999
            scores_tied = score_gap <= TIED_THRESHOLD

            if not scores_tied:
                winner           = best
                selection_method = "SCORE_CLEAR"
                reason           = (
                    f"score_gap({best['score']:.2f}_vs_{second['score']:.2f})"
                    if second is not None
                    else f"only_valid_run(score={best['score']:.2f})"
                )
                counters["score_clear"] += 1
            else:
                winner           = best   # createdTime already in sort key
                selection_method = "SCORE_TIED_RECENCY"
                reason           = (
                    f"tied(gap={score_gap:.2f})"
                    f"_recency({best['created_time'][:10]})"
                )
                counters["score_tied"] += 1

    # ── Read winner CSV if not already loaded ────────────────────────
    if winner.get("csv_text") is None:
        try:
            winner["csv_text"] = read_file(score_service, winner["semantic_file"], SAVE_MODE_SCORE)
        except Exception as e:
            print(f"  [{page_num:>4}] ERROR reading winner CSV: {e} — skip")
            counters["failed"] += 1
            continue

    if winner["csv_text"] is None:
        print(f"  [{page_num:>4}] ERROR: winner CSV is None — skip")
        counters["failed"] += 1
        continue

    # ── Read OCR file ────────────────────────────────────────────────
    ocr_text       = ""
    ocr_file_found = False
    if winner.get("ocr_file"):
        try:
            ocr_text       = read_file(score_service, winner["ocr_file"], SAVE_MODE_SCORE) or ""
            ocr_file_found = bool(ocr_text)
        except Exception:
            pass

    # ── Save page outputs ────────────────────────────────────────────
    clean_csv_name = f"page_{page_num}_semantic.csv"
    clean_ocr_name = f"page_{page_num}_ocr.txt"
    page_out_dir   = LOCAL_OUTPUT_SCORE_DIR / f"page_{page_num}"

    if SAVE_MODE_SCORE in ("local", "both"):
        page_out_dir.mkdir(parents=True, exist_ok=True)
        (page_out_dir / clean_csv_name).write_text(
            winner["csv_text"], encoding="utf-8"
        )
        if ocr_text:
            (page_out_dir / clean_ocr_name).write_text(
                ocr_text, encoding="utf-8"
            )

    if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
        page_drive_folder = get_or_create_folder(
            score_service, f"page_{page_num}", score_drive_output_id
        )
        upload_text(score_service, winner["csv_text"],
                    clean_csv_name, page_drive_folder)
        if ocr_text:
            upload_text(score_service, ocr_text,
                        clean_ocr_name, page_drive_folder)

    # ── Log rows ─────────────────────────────────────────────────────
    winner_filename  = winner["semantic_file"]["name"]
    llm_info         = llm_lookup.get(page_name, {})
    llm_winner       = llm_info.get("llm_winner", "unknown")
    llm_winner_score = llm_info.get("llm_winner_score", None)
    agree            = (
        (winner_filename == llm_winner)
        if llm_winner not in ("unknown", "", None)
        else None
    )
    score_gap_vs_llm = None
    if llm_winner_score is not None:
        try:
            score_gap_vs_llm = round(winner["score"] - float(llm_winner_score), 2)
        except Exception:
            pass

    score_log_rows.append({
        "page":             page_name,
        "page_num":         page_num,
        "n_runs_found":     len(scored_runs),
        "n_valid_runs":     n_valid,
        "winner_filename":  winner_filename,
        "selection_method": selection_method,
        "winner_score":     round(winner["score"], 2),
        "scores_tied":      scores_tied,
        "reason":           reason,
        "all_scores":       all_scores_str,
        "single_run_flag":  single_run_flag,
        "ocr_file_found":   ocr_file_found,
        "timestamp":        datetime.now().isoformat(),
    })

    comparison_rows.append({
        "page":               page_name,
        "page_num":           page_num,
        "score_winner":       winner_filename,
        "score_winner_score": round(winner["score"], 2),
        "llm_winner":         llm_winner,
        "llm_winner_score":   llm_winner_score,
        "agree":              agree,
        "score_gap":          score_gap_vs_llm,
        "scores_tied":        scores_tied,
        "selection_method":   selection_method,
    })

    elapsed = time.time() - t_start
    print(
        f"  [{page_num:>4}] {winner_filename:<35} "
        f"score={winner['score']:.1f} "
        f"{'TIED' if scores_tied else '    '} "
        f"agree={'Y' if agree else ('N' if agree is False else '?')} "
        f"[{elapsed:.0f}s]"
    )

# SAVE LOGS AND STATS

print("\nSaving logs and stats ...")
# Merge new rows with preserved existing rows, sort by page_num
score_log_df  = pd.DataFrame(_existing_log_rows + score_log_rows)
comparison_df = pd.DataFrame(_existing_comp_rows + comparison_rows)
if not score_log_df.empty and "page_num" in score_log_df.columns:
    score_log_df = score_log_df.sort_values("page_num").reset_index(drop=True)
if not comparison_df.empty and "page_num" in comparison_df.columns:
    comparison_df = comparison_df.sort_values("page_num").reset_index(drop=True)

score_log_path  = LOCAL_OUTPUT_SCORE_DIR / "score_selection_log.csv"
comparison_path = LOCAL_OUTPUT_SCORE_DIR / "score_vs_llm_comparison.csv"

# score_log_rows and comparison_rows already contain old rows (pre-populated in resume mode) plus new rows from this run so we always write the full combined set
if SAVE_MODE_SCORE in ("local", "both"):
    score_log_df.to_csv(score_log_path, index=False)
    comparison_df.to_csv(comparison_path, index=False)
    print(f"  score_selection_log.csv   → {score_log_path} "
          f"({len(score_log_df)} total rows)")
    print(f"  score_vs_llm_comparison.csv → {comparison_path} "
          f"({len(comparison_df)} total rows)")

if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
    for df_out, fname in [
        (score_log_df,  "score_selection_log.csv"),
        (comparison_df, "score_vs_llm_comparison.csv"),
    ]:
        buf = io.StringIO()
        df_out.to_csv(buf, index=False)
        upload_text(score_service, buf.getvalue(), fname, score_drive_output_id)
        print(f"  ↑ {fname} ({len(df_out)} total rows)")

# Stats report
n_total           = len(score_log_rows)
n_agree           = int(comparison_df["agree"].sum()) if "agree" in comparison_df else 0
n_disagree        = int((comparison_df["agree"] == False).sum())
n_unknown         = int(comparison_df["agree"].isna().sum())
n_tied            = int(score_log_df["scores_tied"].sum())
n_tied_disagree   = int(comparison_df[
    comparison_df["scores_tied"] & (comparison_df["agree"] == False)
].shape[0])
n_meaningful_disagree = n_disagree - n_tied_disagree

stats_lines = [
    "# Score-Only Version Selector — Statistics Report",
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"READ_CSV_FOR_SCORING: {READ_CSV_FOR_SCORING}",
    f"TIED_THRESHOLD: {TIED_THRESHOLD}",
    "",
    "---",
    "",
    "## Selection Method Breakdown",
    "",
    "| Method | Pages |",
    "|--------|-------|",
    f"| Single run (no choice) | {counters['single']} |",
    f"| Score clear winner (gap > {TIED_THRESHOLD}) | {counters['score_clear']} |",
    f"| Score tied — recency tiebreaker | {counters['score_tied']} |",
    f"| All excluded — fallback | {counters['excluded_fallback']} |",
    f"| Failed (no files) | {counters['failed']} |",
    f"| **Total** | **{n_total}** |",
    "",
    "---",
    "",
    "## Comparison with LLM Selection",
    "",
    "| Metric | Value |",
    "|--------|-------|",
    f"| Pages compared | {n_total - n_unknown} |",
    f"| Agree | {n_agree} ({100*n_agree/max(n_total-n_unknown,1):.1f}%) |",
    f"| Disagree total | {n_disagree} ({100*n_disagree/max(n_total-n_unknown,1):.1f}%) |",
    f"| Disagree where tied | {n_tied_disagree} |",
    f"| **Meaningful disagreements** | **{n_meaningful_disagree}** |",
    f"| LLM winner unknown | {n_unknown} |",
    "",
    "---",
    "",
    "## Meaningful Disagreements (score clear, agree=False)",
    "",
    "| Page | Score winner | Score | LLM winner | LLM score | Gap |",
    "|------|-------------|-------|-----------|-----------|-----|",
]

meaningful_df = comparison_df[
    (comparison_df["agree"] == False) & (~comparison_df["scores_tied"])
].sort_values("page_num")
for _, row in meaningful_df.iterrows():
    stats_lines.append(
        f"| {row['page']} | {row['score_winner']} | {row['score_winner_score']} "
        f"| {row['llm_winner']} | {row['llm_winner_score']} | {row['score_gap']} |"
    )

stats_lines += [
    "",
    "---",
    "",
    "## Pages Where All Runs Were Excluded",
    "",
]
excl_pages = score_log_df[score_log_df["selection_method"] == "EXCLUDED_FALLBACK"]
if excl_pages.empty:
    stats_lines.append("None.")
else:
    for _, row in excl_pages.iterrows():
        stats_lines.append(
            f"- {row['page']} | chosen: {row['winner_filename']} "
            f"| score: {row['winner_score']}"
        )

stats_lines += ["", "---", "", "_End of report_"]
stats_text = "\n".join(stats_lines)

stats_path = LOCAL_OUTPUT_SCORE_DIR / "score_statistics_report.md"
if SAVE_MODE_SCORE in ("local", "both"):
    stats_path.write_text(stats_text, encoding="utf-8")
if SAVE_MODE_SCORE in ("drive", "both") and score_drive_output_id:
    upload_text(score_service, stats_text,
                "score_statistics_report.md", score_drive_output_id)
    print(f"  ↑ score_statistics_report.md")

# ── Final summary ────────────────────────────────────────────────────────
total_elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"  DONE  ({total_elapsed/60:.1f} min)")
print(f"{'='*60}")
print(f"  Single run         : {counters['single']}")
print(f"  Score clear        : {counters['score_clear']}")
print(f"  Score tied         : {counters['score_tied']}")
print(f"  Excluded fallback  : {counters['excluded_fallback']}")
print(f"  Failed             : {counters['failed']}")
print(f"\n  Agreement with LLM : {n_agree}/{n_total-n_unknown} "
      f"({100*n_agree/max(n_total-n_unknown,1):.1f}%)")
print(f"  Meaningful disagree: {n_meaningful_disagree}")
print(f"  Tied disagree      : {n_tied_disagree}")
print(f"\n  Output: {DRIVE_OUTPUT_SCORE_FOLDER}/ on Drive")

  SCORE-ONLY VERSION SELECTOR
  SAVE_MODE_SCORE      : drive
  READ_CSV_FOR_SCORING : True
  Output folder        : clean_pages
  Drive output folder: clean_pages ✓

Loading eval data ...
  Eval loaded from Drive
  1619 semantic rows | 564 pages

Loading LLM selection log ...
  LLM log loaded from Drive: 571 rows

Listing page folders ...
  85 page folders found

Running score-only selection ...
  Preserving 485 existing log rows outside this run's scope
  [  41] page_41_semantic_2.csv              score=73.4      agree=N [9s]
  [ 130] FLAG: only one run found (should not happen)
  [ 130] page_130_semantic_1.csv             score=75.1      agree=Y [17s]
  [ 161] FLAG: only one run found (should not happen)
  [ 161] page_161_semantic_1.csv             score=65.0      agree=Y [24s]
  [ 170] page_170_semantic_1.csv             score=67.0      agree=Y [32s]
  [ 320] page_320_semantic_2.csv             score=73.6 TIED agree=N [40s]
  [ 367] page_367_semantic_1.csv             score=74.7    

## Comparison: score-only vs LLM-arbitration
Ground-truth agreement check between the two methods.


In [ ]:

# Compares the two designs' outputs against ground truth


SCORE_VS_LLM_CSV = Path(CLEAN_PAGES_DIR) / "score_vs_llm_comparison.csv"
EVAL_RESULTS_GLOB = "evaluation/**/*_results.csv"
GT_METRICS = ["char_accuracy", "bow_similarity", "semantic_precision", "semantic_recall", "semantic_f1"]

sel = pd.read_csv(SCORE_VS_LLM_CSV)

# Collect every (Page, Run) row that has ground-truth metrics, across all eval result files. 
# If more than one file evaluated the same (Page, Run) pair, the alphabetically last file (by path, which sorts by timestamp) wins.
gt_frames = []
for f in sorted(glob.glob(EVAL_RESULTS_GLOB, recursive=True)):
    try:
        df = pd.read_csv(f)
    except Exception:
        continue
    if "char_accuracy" not in df.columns or "Page" not in df.columns or "Run" not in df.columns:
        continue
    df = df[df["char_accuracy"].notna()].copy()
    if df.empty:
        continue
    df["source_file"] = f
    gt_frames.append(df[["Page", "Run", "source_file"] + GT_METRICS])

gt_all = pd.concat(gt_frames, ignore_index=True)
gt_all["Page"] = gt_all["Page"].str.replace(r"\.jpg$", "", regex=True)
gt_all = gt_all.drop_duplicates(subset=["Page", "Run"], keep="last")

gt_pages = sorted(gt_all["Page"].unique())
print(f"{len(gt_pages)} ground-truth pages found across {len(gt_frames)} eval result file(s): {gt_pages}")

# For each ground-truth page, look up which run each method picked, then that run's ground-truth metrics.
rows = []
missing = []
for p in gt_pages:
    srow = sel[sel["page"] == p]
    if srow.empty:
        missing.append((p, "not in score_vs_llm_comparison.csv"))
        continue
    srow = srow.iloc[0]
    score_gt = gt_all[(gt_all["Page"] == p) & (gt_all["Run"] == srow["score_winner"])]
    llm_gt = gt_all[(gt_all["Page"] == p) & (gt_all["Run"] == srow["llm_winner"])]
    if score_gt.empty or llm_gt.empty:
        missing.append((p, f"no GT row for score_winner={srow['score_winner']} or llm_winner={srow['llm_winner']}"))
        continue
    row = {"page": p, "score_winner": srow["score_winner"], "llm_winner": srow["llm_winner"]}
    for m in GT_METRICS:
        row[f"score_{m}"] = score_gt.iloc[0][m]
        row[f"llm_{m}"] = llm_gt.iloc[0][m]
    rows.append(row)

if missing:
    print(f"\n{len(missing)} ground-truth page(s) skipped:")
    for p, reason in missing:
        print(f"  {p}: {reason}")

detail_df = pd.DataFrame(rows)
print(f"\n{len(detail_df)} ground-truth pages compared.")

summary_rows = []
for m in GT_METRICS:
    summary_rows.append({
        "metric": m,
        "score_only_mean": detail_df[f"score_{m}"].mean(),
        "llm_arbitration_mean": detail_df[f"llm_{m}"].mean(),
        "score_only_median": detail_df[f"score_{m}"].median(),
        "llm_arbitration_median": detail_df[f"llm_{m}"].median(),
    })
summary_df = pd.DataFrame(summary_rows)

print("\n=== Score-only vs LLM-arbitration, ground-truth metrics of the selected run ===\n")
print(summary_df.round(2).to_string(index=False))

detail_df.to_csv("score_vs_llm_ground_truth_detail.csv", index=False)
summary_df.to_csv("score_vs_llm_ground_truth_summary.csv", index=False)
print(f"\nSaved: score_vs_llm_ground_truth_detail.csv ({len(detail_df)} pages)")
print("Saved: score_vs_llm_ground_truth_summary.csv")


15 ground-truth pages found across 12 eval result file(s): ['page_10', 'page_103', 'page_110', 'page_14', 'page_151', 'page_167', 'page_36', 'page_4', 'page_42', 'page_45', 'page_58', 'page_60', 'page_63', 'page_86', 'page_87']

1 ground-truth page(s) skipped:
  page_4: no GT row for score_winner=page_4_semantic_3.csv or llm_winner=page_4_semantic_1.csv

14 ground-truth pages compared.

=== Score-only vs LLM-arbitration, ground-truth metrics of the selected run ===

            metric  score_only_mean  llm_arbitration_mean  score_only_median  llm_arbitration_median
     char_accuracy            87.51                 84.85              95.37                   94.75
    bow_similarity            67.46                 63.65              74.04                   72.73
semantic_precision            65.02                 56.26              65.28                   59.85
   semantic_recall            63.07                 53.31              64.48                   57.56
       semantic_f1      